<a href="https://colab.research.google.com/github/watanabe3tipapa/GoogleColaboratory/blob/main/verification/houritsu_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# (検証) 日本の法令を Git の履歴にする

e-Gov 法令検索 API v2 から法令データを取得し、
[Zenn 記事『日本の全法令を git の履歴にしてみた』](https://zenn.dev/koheiwada/articles/houritsu-japanese-laws-as-git) の手法で Git リポジトリを構築します。

**構成:**
- 1 法令 = 1 Markdown ファイル（メタデータは frontmatter）
- 1 コミット = 1 本の改正法の施行
- コミット日付 = 施日
- 1970年より前の施行日は 1970-01-02 に寄せ、本当の日付は frontmatter に記録
- 公布済みで未施行の改正は `upcoming` ブランチに配置

## 1. 準備

In [4]:
import os
import subprocess
import json
import urllib.request
import urllib.parse
import urllib.error
from datetime import datetime
from pathlib import Path
import time
import re

BASE_URL = "https://laws.e-gov.go.jp/api/2"
REPO_DIR = Path("laws_repo")

# Git の存在確認
result = subprocess.run(["git", "--version"], capture_output=True, text=True)
print(result.stdout.strip())

git version 2.43.0


## 2. Git リポジトリの初期化

In [5]:
# リポジトリディレクトリの作成・初期化
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True)
    subprocess.run(["git", "init"], cwd=REPO_DIR, check=True)

    # Gitユーザー情報の設定（Colab環境などの未設定エラーを防止）
    subprocess.run(["git", "config", "user.email", "you@example.com"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.name", "Your Name"], cwd=REPO_DIR, check=True)

    # 初期コミット（空）
    subprocess.run(["git", "commit", "--allow-empty", "-m", "Initial commit"], cwd=REPO_DIR, check=True)
    print("リポジトリを初期化しました")
else:
    print("リポジトリは既に存在します")

リポジトリを初期化しました


In [6]:
import shutil
from pathlib import Path

# laws_repo ディレクトリをきれいに削除する
repo_dir = Path("laws_repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
    print("laws_repo ディレクトリを削除しました。再び上の『2. Git リポジトリの初期化』セルを実行してください。")
else:
    print("laws_repo ディレクトリは存在しません。そのまま上のセルを実行できます。")

laws_repo ディレクトリを削除しました。再び上の『2. Git リポジトリの初期化』セルを実行してください。


## 3. e-Gov API クライアント

In [7]:
def api_get(endpoint, params=None):
    """API から JSON を取得する"""
    url = f"{BASE_URL}{endpoint}"
    if params:
        query = urllib.parse.urlencode(params, doseq=True)
        url = f"{url}?{query}"
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as res:
            return json.loads(res.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return None  # 取得結果0件など
        raise

def get_all_laws(law_type=None, limit=1000):
    """全法令一覧を取得"""
    laws = []
    offset = 0
    while True:
        params = {"limit": limit, "offset": offset}
        if law_type:
            params["law_type"] = law_type
        data = api_get("/laws", params)
        if not data or not data.get("laws"):
            break
        laws.extend(data["laws"])
        if "next_offset" not in data or data["next_offset"] is None:
            break
        offset = data["next_offset"]
        time.sleep(0.5)
    return laws

def get_law_revisions(law_id_or_num):
    """法令の改正履歴を取得"""
    data = api_get(f"/law_revisions/{law_id_or_num}")
    if not data:
        return []
    return data.get("revisions", [])

def get_law_data(law_id_or_num_or_revision_id, asof=None, json_format="light"):
    """法令本文を取得"""
    params = {"json_format": json_format}
    if asof:
        params["asof"] = asof
    return api_get(f"/law_data/{law_id_or_num_or_revision_id}", params)

print("API クライアントを定義しました")

API クライアントを定義しました


## 4. Markdown 変換関数

In [8]:
def law_data_to_markdown(law_data):
    """
    law_data の JSON（light 形式）を Markdown 変換する。
    簡易実装: LawTitle, MainProvision, SupplProvision を再帰的に処理。
    """
    info = law_data.get("law_info", {})
    rev = law_data.get("revision_info", {})
    text = law_data.get("law_full_text", {})

    # frontmatter
    frontmatter = {"---": None}
    frontmatter_lines = ["---"]
    frontmatter_lines.append(f"law_id: {info.get('law_id', '')}")
    frontmatter_lines.append(f"law_num: {info.get('law_num', '')}")
    frontmatter_lines.append(f"law_type: {info.get('law_type', '')}")
    frontmatter_lines.append(f"law_title: {rev.get('law_title', '')}")
    frontmatter_lines.append(f"law_revision_id: {rev.get('law_revision_id', '')}")
    frontmatter_lines.append(f"amendment_enforcement_date: {rev.get('amendment_enforcement_date', '')}")
    frontmatter_lines.append(f"current_revision_status: {rev.get('current_revision_status', '')}")
    frontmatter_lines.append(f"promulgation_date: {info.get('promulgation_date', '')}")
    frontmatter_lines.append("---")

    md = ["\n".join(frontmatter_lines)]

    # タイトル
    law_title = text.get("LawTitle", rev.get("law_title", ""))
    if isinstance(law_title, dict):
        law_title = law_title.get("#text", rev.get("law_title", ""))
    md.append(f"# {law_title}\n")

    # 目次
    toc = text.get("TOC", {})
    if toc:
        md.append("## 目次\n")
        md.append(convert_toc(toc))

    # 本文（本則）
    main = text.get("MainProvision", {})
    if main:
        md.append(convert_main_provision(main))

    # 附則
    suppl = text.get("SupplProvision", [])
    if suppl:
        if not isinstance(suppl, list):
            suppl = [suppl]
        for s in suppl:
            md.append(convert_suppl_provision(s))

    return "\n".join(md)

def convert_toc(toc):
    lines = []
    items = toc.get("TOCChapter", []) or toc.get("TOCArticle", []) or toc.get("TOC", [])
    if not isinstance(items, list):
        items = [items]
    for item in items:
        if isinstance(item, dict):
            title = item.get("Title", "")
            if isinstance(title, dict):
                title = title.get("#text", "")
            lines.append(f"- {title}")
    return "\n".join(lines) + "\n"

def convert_main_provision(main):
    lines = []
    chapters = main.get("Chapter", [])
    if not isinstance(chapters, list):
        chapters = [chapters]
    for ch in chapters:
        lines.append(convert_chapter(ch))
    # Chapter がない場合は Article 直接
    articles = main.get("Article", [])
    if not isinstance(articles, list):
        articles = [articles]
    for art in articles:
        lines.append(convert_article(art))
    return "\n".join(lines)

def convert_chapter(ch):
    lines = []
    title = ch.get("Title", "")
    if isinstance(title, dict):
        title = title.get("#text", "")
    lines.append(f"## {title}\n")
    articles = ch.get("Article", [])
    if not isinstance(articles, list):
        articles = [articles]
    for art in articles:
        lines.append(convert_article(art))
    return "\n".join(lines)

def convert_article(art):
    lines = []
    caption = art.get("ArticleCaption", "")
    if isinstance(caption, dict):
        caption = caption.get("#text", "")
    num = art.get("Num", "")
    title = art.get("ArticleTitle", "")
    if isinstance(title, dict):
        title = title.get("#text", "")
    header = f"### {title}"
    if caption:
        header += f"（{caption}）"
    lines.append(header + "\n")

    paragraphs = art.get("Paragraph", [])
    if not isinstance(paragraphs, list):
        paragraphs = [paragraphs]
    for p in paragraphs:
        lines.append(convert_paragraph(p))
    return "\n".join(lines)

def convert_paragraph(p):
    num = p.get("Num", "")
    sentence = p.get("ParagraphSentence", {})
    text = extract_sentences(sentence)
    if num:
        return f"{num}. {text}\n"
    return f"{text}\n"

def extract_sentences(sentence_obj):
    if not sentence_obj:
        return ""
    sentences = sentence_obj.get("Sentence", [])
    if isinstance(sentences, str):
        return sentences
    if isinstance(sentences, dict):
        return sentences.get("#text", "")
    if isinstance(sentences, list):
        texts = []
        for s in sentences:
            if isinstance(s, str):
                texts.append(s)
            elif isinstance(s, dict):
                texts.append(s.get("#text", ""))
        return "".join(texts)
    return str(sentence_obj)

def convert_suppl_provision(suppl):
    lines = []
    num = suppl.get("Num", "")
    extract = suppl.get("Extract", "")
    header = "## 附則"
    if num:
        header += f"（{num}）"
    if extract:
        header += f" {extract}"
    lines.append(header + "\n")
    articles = suppl.get("Article", [])
    if not isinstance(articles, list):
        articles = [articles]
    for art in articles:
        lines.append(convert_article(art))
    return "\n".join(lines)

print("Markdown 変換関数を定義しました")

Markdown 変換関数を定義しました


## 5. Git 操作ヘルパー

In [9]:
def git_commit(repo_dir, message, date_iso, trailers=None, files=None):
    """
    Git コミットを作成する。
    1970-01-02 より前の日付は 1970-01-02 に寄せ、本当の日付はトレーラーに入れる。
    """
    min_date = datetime(1970, 1, 2)
    commit_date = datetime.fromisoformat(date_iso.replace("Z", "+00:00"))
    actual_date = date_iso

    if commit_date < min_date:
        commit_date = min_date

    date_str = commit_date.strftime("%Y-%m-%dT%H:%M:%S+09:00")

    env = os.environ.copy()
    env["GIT_AUTHOR_DATE"] = date_str
    env["GIT_COMMITTER_DATE"] = date_str

    # トレーラー追加
    full_message = message
    if trailers:
        for key, value in trailers.items():
            full_message += f"\n{key}: {value}"
    if commit_date == min_date and actual_date != date_iso:
        full_message += f"\nEnforced: {actual_date}"

    if files:
        subprocess.run(["git", "add", "--"] + files, cwd=repo_dir, check=True)
    else:
        subprocess.run(["git", "add", "-A"], cwd=repo_dir, check=True)

    cmd = ["git", "commit", "-m", full_message]
    result = subprocess.run(cmd, cwd=repo_dir, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        if "nothing to commit" in result.stderr.lower() or "nothing added to commit" in result.stderr.lower():
            return False
        print("STDERR:", result.stderr)
        raise RuntimeError(f"git commit failed: {result.stderr}")
    return True

def git_checkout_branch(repo_dir, branch_name, orphan=False):
    """ブランチを切り替え（存在しなければ作成）"""
    result = subprocess.run(
        ["git", "branch", "--list", branch_name],
        cwd=repo_dir, capture_output=True, text=True
    )
    if orphan:
        subprocess.run(["git", "checkout", "--orphan", branch_name], cwd=repo_dir, check=True)
        subprocess.run(["git", "rm", "-rf", "."], cwd=repo_dir, capture_output=True)
    else:
        if result.stdout.strip():
            subprocess.run(["git", "checkout", branch_name], cwd=repo_dir, check=True)
        else:
            subprocess.run(["git", "checkout", "-b", branch_name], cwd=repo_dir, check=True)

print("Git 操作ヘルパーを定義しました")

Git 操作ヘルパーを定義しました


## 6. 法令ファイルパス生成

In [10]:
def get_law_file_path(law_info):
    """
    法令種別と法令IDからファイルパスを生成する。
    例: laws/act/322/322AC0000000049.md
    """
    law_type = law_info.get("law_type", "unknown").lower()
    law_id = law_info.get("law_id", "")
    if not law_id:
        return Path("laws") / law_type / f"{law_id}.md"
    # 法令IDの先頭3桁をディレクトリ階層にする
    prefix = law_id[:3] if len(law_id) >= 3 else law_id
    return Path("laws") / law_type / prefix / f"{law_id}.md"

print("ファイルパス生成関数を定義しました")

ファイルパス生成関数を定義しました


## 7. 単一法令の履歴構築

In [11]:
def build_law_history(law_summary, dry_run=False):
    """
    1つの法令の改正履歴を取得し、施行日ごとに Markdown を書き出してコミット対象フイルを返す。
    戻り値: list of (file_path, revision_id, enforcement_date, current_revision_status)
    """
    law_id = law_summary["law_info"]["law_id"]
    revisions = get_law_revisions(law_id)
    if not revisions:
        return []

    files_info = []
    for rev in revisions:
        rev_id = rev.get("law_revision_id")
        enforcement_date = rev.get("amendment_enforcement_date")
        status = rev.get("current_revision_status")
        if not rev_id:
            continue
        # 未施行のものは upcoming ブランチ用に分ける
        files_info.append((law_summary, rev_id, enforcement_date, status))
    return files_info

print("build_law_history を定義しました")

build_law_history を定義しました


## 8. 全法令の取得と履歴の集約

In [12]:
# 対象法令種別（必要に応じて絞る）
TARGET_LAW_TYPES = ["Act", "CabinetOrder", "MinisterialOrdinance"]

all_laws = []
for lt in TARGET_LAW_TYPES:
    print(f"取得中: {lt} ...")
    laws = get_all_laws(law_type=lt)
    print(f"  -> {len(laws)} 件")
    all_laws.extend(laws)
    time.sleep(1)

print(f"\n全法令: {len(all_laws)} 件")

取得中: Act ...
  -> 2165 件
取得中: CabinetOrder ...
  -> 2448 件
取得中: MinisterialOrdinance ...
  -> 4450 件

全法令: 9063 件


## 9. 施行日ごとにコミットを集約

In [13]:
from collections import defaultdict

# 各法令の履歴を取得して集約
# commits_map: enforcement_date -> list of (law_summary, revision_id, status)
commits_map = defaultdict(list)
upcoming_items = []

for i, law_summary in enumerate(all_laws):
    law_id = law_summary["law_info"]["law_id"]
    print(f"[{i+1}/{len(all_laws)}] {law_id}")
    try:
        revs = build_law_history(law_summary)
        for _, rev_id, enf_date, status in revs:
            if status == "UnEnforced":
                upcoming_items.append((law_summary, rev_id, enf_date))
            elif enf_date:
                commits_map[enf_date].append((law_summary, rev_id, status))
            else:
                # 施行日がない場合はスキップ or 別途処理
                pass
        time.sleep(0.3)
    except Exception as e:
        print(f"  エラー: {e}")

print(f"\n施行日ごとのコミット数: {len(commits_map)}")
print(f"未施行アイテム数: {len(upcoming_items)}")

[1/9063] 106DF0000000065
[2/9063] 113DF0000000036
[3/9063] 117DF1000000032
[4/9063] 122AC0000000034
[5/9063] 123AC0000000001
[6/9063] 128AC0000000028
[7/9063] 129AC0000000005
[8/9063] 129AC0000000089
[9/9063] 130AC0000000029
[10/9063] 131AC0000000011
[11/9063] 131AC0000000014
[12/9063] 132AC0000000015
[13/9063] 132AC0000000046
[14/9063] 132AC0000000048
[15/9063] 132AC0000000049
[16/9063] 132AC0000000050
[17/9063] 132AC0000000093
[18/9063] 132AC0000000095
[19/9063] 132AC0000000101
[20/9063] 132AC1000000040
[21/9063] 133AC0000000065
[22/9063] 133AC0000000067
[23/9063] 133AC1000000033
[24/9063] 133AC1000000072
[25/9063] 135AC0000000011
[26/9063] 135AC1000000050
[27/9063] 137AC0000000017
[28/9063] 138AC0000000052
[29/9063] 138AC0000000053
[30/9063] 138AC0000000054
[31/9063] 138AC0000000055
[32/9063] 138AC0000000063
[33/9063] 138AC1000000066
[34/9063] 139AC0000000034
[35/9063] 139AC0000000051
[36/9063] 140AC0000000045
[37/9063] 141AC0000000029
[38/9063] 141AC0000000050
[39/9063] 141AC000000

KeyboardInterrupt: 

## 10. main ブランチの構築（施行日順にコミット）

In [14]:
git_checkout_branch(REPO_DIR, "main")

# 施行日でソート
sorted_dates = sorted(commits_map.keys())

for date_iso in sorted_dates:
    items = commits_map[date_iso]
    changed_laws = []
    amendment_law_num = None
    promulgation_date = None

    for law_summary, rev_id, status in items:
        law_id = law_summary["law_info"]["law_id"]
        law_title = law_summary["revision_info"].get("law_title", "")
        # 法令データを取得
        try:
            data = get_law_data(rev_id, json_format="light")
            if not data:
                continue
            md = law_data_to_markdown(data)
            file_path = get_law_file_path(data["law_info"])
            full_path = REPO_DIR / file_path
            full_path.parent.mkdir(parents=True, exist_ok=True)
            full_path.write_text(md, encoding="utf-8")
            changed_laws.append(str(file_path))
            # 改正法令情報の抽出（revision_info.amendment_law_num など）
            rev_info = data.get("revision_info", {})
            if not amendment_law_num:
                amendment_law_num = rev_info.get("amendment_law_num", "")
            if not promulgation_date:
                promulgation_date = rev_info.get("promulgation_date", "")
        except Exception as e:
            print(f"  取得失敗 {law_id}: {e}")
        time.sleep(0.2)

    if not changed_laws:
        continue

    # コミットメッセージ構築
    # 改正法の題名を取得するには law_revisions の情報を使う
    first_law = items[0][0]
    revs = get_law_revisions(first_law["law_info"]["law_id"])
    amendment_title = ""
    for r in revs:
        if r.get("amendment_enforcement_date") == date_iso:
            amendment_title = r.get("amendment_law_title", "")
            amendment_law_num = r.get("amendment_law_num", amendment_law_num)
            promulgation_date = r.get("promulgation_date", promulgation_date)
            break

    commit_msg = f"{amendment_title or '法令改正'} 施行"
    trailers = {}
    if amendment_law_num:
        trailers["改正法令"] = amendment_law_num
    if promulgation_date:
        trailers["公布日"] = promulgation_date
    trailers["施行日"] = date_iso

    # 変更された法令一覧
    law_names = []
    for law_summary, _, _ in items:
        title = law_summary["revision_info"].get("law_title", "")
        law_num = law_summary["law_info"].get("law_num", "")
        if title:
            law_names.append(f"- {title}（{law_num}）" if law_num else f"- {title}")
    if law_names:
        trailers["変更された法令"] = "\n" + "\n".join(law_names)

    committed = git_commit(REPO_DIR, commit_msg, date_iso, trailers=trailers, files=changed_laws)
    if committed:
        print(f"[{date_iso}] {commit_msg} ({len(changed_laws)} files)")
    else:
        print(f"[{date_iso}] 変更なし")

print("main ブランチの構築完了")

FileNotFoundError: [Errno 2] No such file or directory: PosixPath('laws_repo')

## 11. upcoming ブランチの構築（未施行の改正）

In [ ]:
git_checkout_branch(REPO_DIR, "upcoming", orphan=True)

# main ブランチの最新ファイルをチェックアウトして上書き
subprocess.run(["git", "checkout", "main", "--", "."], cwd=REPO_DIR, check=True)

for law_summary, rev_id, enf_date in upcoming_items:
    law_id = law_summary["law_info"]["law_id"]
    try:
        data = get_law_data(rev_id, json_format="light")
        if not data:
            continue
        md = law_data_to_markdown(data)
        file_path = get_law_file_path(data["law_info"])
        full_path = REPO_DIR / file_path
        full_path.parent.mkdir(parents=True, exist_ok=True)
        full_path.write_text(md, encoding="utf-8")
        subprocess.run(["git", "add", str(file_path)], cwd=REPO_DIR, check=True)
        print(f"upcoming: {file_path} -> {enf_date}")
    except Exception as e:
        print(f"  失敗 {law_id}: {e}")
    time.sleep(0.2)

# upcoming ブランチにコミット
subprocess.run(
    ["git", "commit", "-m", "upcoming: 公布済み未施行の改正を反映"],
    cwd=REPO_DIR, capture_output=True
)
print("upcoming ブランチの構築完了")

## 12. 使い方の例

In [ ]:
# 個人情報保護法の履歴を見る例
import subprocess

# ファイルパスは法令IDから生成
# 例: 個人情報保護法（415AC0000000057）
example_law_id = "415AC0000000057"
example_path = f"laws/act/415/{example_law_id}.md"

result = subprocess.run(
    ["git", "log", "--format=%ad %s", "--date=short", "--", example_path],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[:2000] if result.stdout else "該当なし")

In [ ]:
# 特定条文の変遷を追う例（git log -L）
# 注意: git log -L はファイル内の行範囲を指定するため、Markdown 変換後の形式に依存
example_article = "第十六条"
result = subprocess.run(
    ["git", "log", "-L", f"/^##*  *{example_article}/,/^##*  *第十七条/:{example_path}"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[:2000] if result.stdout else "該当なし or エラー")
if result.stderr:
    print("STDERR:", result.stderr[:500])

## 付録: 個人情報保護法の例

### A. 法令IDの検索

In [ ]:
# 個人情報保護法の法令IDを検索する
import urllib.request
import urllib.parse
import json

BASE_URL = "https://laws.e-gov.go.jp/api/2"

def api_get(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"
    if params:
        query = urllib.parse.urlencode(params, doseq=True)
        url = f"{url}?{query}"
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as res:
        return json.loads(res.read().decode("utf-8"))

# 法令名で検索
results = api_get("/laws", {"law_title": "個人情報の保護に関する法律", "limit": 5})
for law in results.get("laws", []):
    info = law["law_info"]
    rev = law["revision_info"]
    print(f"法令ID: {info['law_id']}")
    print(f"法令番号: {info['law_num']}")
    print(f"法令種別: {info['law_type']}")
    print(f"題名: {rev['law_title']}")
    print(f"公布日: {info['promulgation_date']}")
    print(f"現施行日: {rev['amendment_enforcement_date']}")
    print(f"状態: {rev['current_revision_status']}")
    print("---")


### B. 改正履歴の取得

In [ ]:
# 個人情報保護法の改正履歴を取得する
LAW_ID = "415AC0000000057"  # 個人情の保護に関する法律

revs = api_get(f"/law_revisions/{LAW_ID}")
print(f"改正履歴: {len(revs['revisions'])} 件\n")

for r in revs['revisions']:
    rev_id = r['law_revision_id']
    title = r['law_title']
    enf = r['amendment_enforcement_date']
    status = r['current_revision_status']
    amend_title = r.get('amendment_law_title', '')
    amend_num = r.get('amendment_law_num', '')
    print(f"[{status}] {enf} | {rev_id}")
    if amend_title:
        print(f"  改正法: {amend_title}")
    if amend_num:
        print(f"  改正法令番号: {amend_num}")
    print()


### C. 特定改正版の本文を Markdown で取得

In [ ]:
# 例: 令和8年法律第60号による改正版（2026-07-17 施行）の本文を取得
# 改正履歴から該当する revision_id を探す
target_rev_id = None
for r in revs['revisions']:
    if r['amendment_enforcement_date'] == '2026-07-17':
        target_rev_id = r['law_revision_id']
        break

if target_rev_id:
    data = api_get(f"/law_data/{target_rev_id}?json_format=light")
    print(f"取得した法令: {data['revision_info']['law_title']}")
    print(f"施行日: {data['revision_info']['amendment_enforcement_date']}")
    print(f"改正法: {data['revision_info'].get('amendment_law_title', 'N/A')}")
    print(f"改正法令番号: {data['revision_info'].get('amendment_law_num', 'N/A')}")
    print("\n--- 本文（先頭3000文字） ---\n")
    # Markdown 変換関数を使う（上のセルで定義済み）
    md = law_data_to_markdown(data)
    print(md[:3000])
else:
    print("該当する改正版が見つかりませんでした")


### D. Git リポジトリでの確認例

In [ ]:
import subprocess

REPO_DIR = "laws_repo"
LAW_PATH = "laws/act/415/415AC0000000057.md"

# 1. 個人情報保護法の改正履歴を git log で見る
print("=== git log: 個人情報保護法の改正履歴 ===\n")
result = subprocess.run(
    ["git", "log", "--format=%ad %s", "--date=short", "--", LAW_PATH],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[:2000] if result.stdout else "該当なし")


In [ ]:
# 2. 第十六条の変遷を git log -L で追う
print("=== git log -L: 第十六条の変遷 ===\n")
result = subprocess.run(
    ["git", "log", "-L", f"/^##*  *第十六条/,/^##*  *第十七条/:{LAW_PATH}"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[:3000] if result.stdout else "該当なし or エラー")
if result.stderr:
    print("\nSTDERR:", result.stderr[:500])


In [ ]:
# 3. 特定コミットの差分を見る（例: 2026-07-17 の改正）
print("=== git show: 2026-07-17 の改正の差分 ===\n")
# まず該当コミットを探す
result = subprocess.run(
    ["git", "log", "--format=%H %ad %s", "--date=short", "--grep", "2026-07-17", "-1"],
    cwd=REPO_DIR, capture_output=True, text=True
)
if result.stdout.strip():
    commit_hash = result.stdout.strip().split()[0]
    print(f"コミット: {commit_hash}\n")
    diff_result = subprocess.run(
        ["git", "show", "--stat", commit_hash],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    print(diff_result.stdout[:2000])
else:
    print("該当コミットが見つかりませんでした")


In [ ]:
# 4. 個人情報保護法のファイルを blame して、各条文が最後に変更されたコミットを見る
print("=== git blame: 第十六条の最終変更コミット ===\n")
result = subprocess.run(
    ["git", "blame", "-L", "/^### 第十六条/,/^### 第十七条/", LAW_PATH],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[:2000] if result.stdout else "該当なし or エラー")
if result.stderr:
    print("STDERR:", result.stderr[:500])


### E. 複数法令への波及を確認する例

In [ ]:
# 特定の改正法（例: 令和8年法律第60号）がどれだけの法令に影響を与えたか確認
print("=== 令和8年法律第60号の影響範囲 ===\n")
result = subprocess.run(
    ["git", "log", "--all-match", "--grep="改正法令: 令和八年法律第六十号"", "--format=%ad %s", "--date=short"],
    cwd=REPO_DIR, capture_output=True, text=True
)
if result.stdout.strip():
    lines = result.stdout.strip().split("\n")
    print(f"影響を受けたコミット数: {len(lines)}")
    print("\n先頭10件:")
    for line in lines[:10]:
        print(line)
else:
    # grep が複数語に対応していない場合の代替
    result2 = subprocess.run(
        ["git", "log", "--format=%H %s", "--grep="令和八年法律第六十号""],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    if result2.stdout.strip():
        lines = result2.stdout.strip().split("\n")
        print(f"該当コミット数: {len(lines)}")
        for line in lines[:10]:
            print(line)
    else:
        print("該コミットが見つかりませんでした")


## 補足・注意点

- **API レート制限**: e-Gov API は認証不要ですが、大量リクエストは避け、適宜 `time.sleep()` を入れてください。
- **データサイズ**: 全法令（約 9,500 件）を処理ると時間がかかります。テスト時は `TARGET_LAW_TYPES` を絞るか、`all_laws[:10]` などで件数を制限してください。
- **Markdown 変換の精度**: 上記の `law_data_to_markdown` は簡易実装です。複雑な構造（テーブル、号の階層、別表など）は完全には対応していません。必要に応じて XML 版（`law_file` API）を使って詳細なパースを行ってください。
- **1970年問題**: Git は Unix 時間（1970-01-01 以降）しかコミット日付として持てません。1970-01-01 未満の施行日は 1970-01-02 に寄せ、`Enforced` トレーラーに本来の日付を記録しています。
- **議案情報**: 記事では衆議院議案情報との紐付けも行っていますが、本 Notebook では省略しています。必要であれば国会会議録 API 等と組み合わせてください。

## リポジトリ

このノートブックは [GoogleColaboratory](https://github.com/watanabe3tipapa/GoogleColaboratory) リポジトリで公開しています。
